# 4B core benchmark

Complete comparison of `no_loc`, integer-coordinate `loc_text` and L40 `loc_embed`, including matched shuffled-coordinate controls.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
evaluation_root = repo_root / "outputs" / "evaluation"

runs = {
    "no_loc": "11442",
    "loc_text integer": "11443",
    "loc_embed L40": "11444",
}
shuffled_runs = {
    "loc_text integer": "11447",
    "loc_embed L40": "11448",
}
condition_order = list(runs)

def read_json(path):
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)

def load_summary(job):
    return read_json(evaluation_root / job / "scored_predictions" / "summary.json")

summaries = {condition: load_summary(job) for condition, job in runs.items()}
shuffled_summaries = {condition: load_summary(job) for condition, job in shuffled_runs.items()}
predictions = {
    condition: pd.read_json(evaluation_root / job / "predictions.jsonl", lines=True)
    for condition, job in runs.items()
}

pd.DataFrame({
    "Condition": condition_order,
    "Evaluation job": [runs[c] for c in condition_order],
    "Samples": [len(predictions[c]) for c in condition_order],
})

## Population check

In [ ]:
reference_ids = set(predictions["no_loc"]["sample_id"].astype(str))
pd.DataFrame([
    {
        "Condition": condition,
        "Rows": len(frame),
        "Unique sample IDs": frame["sample_id"].astype(str).nunique(),
        "Same IDs as no_loc": set(frame["sample_id"].astype(str)) == reference_ids,
    }
    for condition, frame in predictions.items()
])

## Main results

One primary metric per task family. Average rank weights the four task families equally; lower is better.

In [ ]:
def task_row(summary, task_type):
    return next(row for row in summary["by_task_type"] if row["task_type"] == task_type)

def primary_metrics(summary):
    return {
        "Caption BLEU-4": summary["captioning"]["bleu4"],
        "Binary accuracy": task_row(summary, "binary")["accuracy"],
        "MCQ accuracy": task_row(summary, "mcq")["accuracy"],
        "Bounding-box mIoU": task_row(summary, "bounding box")["miou"],
    }

main_results = pd.DataFrame([
    {"Condition": condition, **primary_metrics(summary)}
    for condition, summary in summaries.items()
]).set_index("Condition").reindex(condition_order)
metric_columns = list(main_results.columns)
main_results["Average rank"] = main_results[metric_columns].rank(ascending=False).mean(axis=1)
main_results.style.format("{:.4f}").highlight_max(
    subset=metric_columns, axis=0, props="font-weight: bold"
).highlight_min(
    subset=["Average rank"], axis=0, props="font-weight: bold"
).set_caption("Primary benchmark metrics")

## Difference from no_loc

In [ ]:
delta = main_results.loc[["loc_text integer", "loc_embed L40"], metric_columns].subtract(main_results.loc["no_loc", metric_columns])
limit = delta.abs().to_numpy().max()
delta.style.format("{:+.4f}").background_gradient(cmap="RdYlGn", vmin=-limit, vmax=limit).set_caption("Location condition − no_loc")

## Task-wise results

In [ ]:
category_rows = []
for condition, summary in summaries.items():
    for row in summary["by_task_category"]:
        category_rows.append({"Condition": condition, **row})
category_scores = pd.DataFrame(category_rows)

def category_table(task_type, metric):
    rows = category_scores[category_scores["task_type"] == task_type]
    table = rows.pivot(index="Condition", columns="task_category", values=metric)
    overall = pd.Series({
        condition: task_row(summaries[condition], task_type)[metric]
        for condition in condition_order
    }, name="Overall")
    return table.reindex(condition_order).join(overall)

display(category_table("binary", "accuracy").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Binary accuracy"))
display(category_table("mcq", "accuracy").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("MCQ accuracy"))
display(category_table("bounding box", "miou").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Bounding-box mIoU"))

## Shuffled-coordinate controls

Values are shuffled minus correct. Negative values mean that replacing the true coordinates hurt performance.

In [ ]:
counterfactual_rows = []
for condition in shuffled_runs:
    correct = primary_metrics(summaries[condition])
    shuffled = primary_metrics(shuffled_summaries[condition])
    counterfactual_rows.append({
        "Condition": condition,
        **{metric: shuffled[metric] - correct[metric] for metric in correct},
    })
counterfactual_deltas = pd.DataFrame(counterfactual_rows).set_index("Condition")
limit = counterfactual_deltas.abs().to_numpy().max()
counterfactual_deltas.style.format("{:+.4f}").background_gradient(cmap="RdYlGn", vmin=-limit, vmax=limit).set_caption("Shuffled − correct coordinates")

### Direct-geography MCQs under shuffling

In [ ]:
def category_accuracy(summary, category):
    return next(
        row["accuracy"]
        for row in summary["by_task_category"]
        if row["task_type"] == "mcq" and row["task_category"] == category
    )

geo_rows = []
for condition in shuffled_runs:
    for category in ["country", "climate zone", "season"]:
        correct = category_accuracy(summaries[condition], category)
        shuffled = category_accuracy(shuffled_summaries[condition], category)
        geo_rows.append({
            "Condition": condition,
            "Category": category,
            "Correct": correct,
            "Shuffled": shuffled,
            "Difference": shuffled - correct,
        })
pd.DataFrame(geo_rows).style.format({"Correct": "{:.3f}", "Shuffled": "{:.3f}", "Difference": "{:+.3f}"})

## Full diagnostic tables

In [ ]:
task_type_rows = []
caption_rows = []
for condition, summary in summaries.items():
    for row in summary["by_task_type"]:
        task_type_rows.append({"Condition": condition, **row})
    caption_rows.append({"Condition": condition, **summary["captioning"]})
display(pd.DataFrame(task_type_rows).sort_values(["task_type", "Condition"]).reset_index(drop=True))
display(pd.DataFrame(caption_rows).set_index("Condition").reindex(condition_order).reset_index())
display(category_scores.sort_values(["task_type", "task_category", "Condition"]).reset_index(drop=True))

## Current reading

- Both location conditions improve MCQ accuracy over `no_loc`; `loc_embed` is best.
- `loc_text` has the strongest caption metrics and slightly improves bounding-box mIoU.
- Shuffling strongly damages captioning and MCQ, confirming that both models use their coordinates.